# Reddit Logs to YTMusic Playlists

In [ ]:
# Set Paths

# Path to header .json file for ytmusic api
ytmusic_header_path = '..'
# Path to .tsv reddit search logs
reddit_log_path = '..\\..\\reddit_scraper\\logs'

In [ ]:
import os
import glob
import time
import math

import unicodedata
from datetime import date
import re
import pandas as pd
from IPython.display import display
from fuzzywuzzy import fuzz
from ytmusicapi import YTMusic

## YTMusic API and Functions

In [2]:
def parse_ytmusic_tracks(track_list):
    tracks = pd.DataFrame(track_list)
    tracks['artistId'] = tracks['artists'].dropna().apply(
        lambda x: x[0]['id'])
    tracks['artist'] = tracks['artists'].dropna().apply(lambda x: x[0]['name'])
    tracks['albumId'] = tracks['album'].dropna().apply(lambda x: x['id'])
    tracks['album'] = tracks['album'].dropna().apply(lambda x: x['name'])
    tracks = tracks.drop('thumbnails', axis=1)
    tracks = tracks.drop('artists', axis=1)
    return tracks

def parse_ytmusic_playlist(yt, playlist_meta):
    playlist_meta.pop('thumbnails', None)
    track_list = playlist_meta.pop('tracks', None)
    tracks = parse_ytmusic_tracks(track_list)
    return tracks, playlist_meta
    
ytm = YTMusic(os.path.join(ytmusic_header_path, 'headers_auth.json'))
yt_res_cache = {}
yt_unmatched_cache = {}

## String Helper Functions

In [3]:
def scrub_title(title):
    title = title.lower().strip()
    title = unicodedata.normalize('NFKD', title).encode(
        'ascii', 'ignore').decode()
    title = title.replace('ft.', 'feat.')
    is_album = False
    for k in ['full album', '(album)', 'album stream']:
        if k in title:
            is_album = True
    for k in [
        '(official', '[official', 'official video', '[official', '[unoffical', '[free', '(free',
        '(explicit', '[video', '(video', '(music', '(nsfw', '(unoffical', '(live', '[live',
        '(video', 'music video', 'live video', '(original', '(lyric', 'lyric video', 
        '[full', '(full', '(album)', 'album stream', 'album review', '[thissongissick',
        '(prod', 'prod.', 'produced by', '[leak', '(from', '(lofi hip']:
        if k in title:
            title = title.split(k)[0]
    for k in ['[hd]', '[hq]', 'hd', 'hq']:
        if k in title:
            title=title.replace(k, '')
    return title, is_album

def strip_non_alphanumeric(value):
    value = str(re.sub('[^\\w\\s-]', ' ', str(value)))
    value = str(re.sub('[-\\s]+', ' ', str(value)))
    return value.strip()    

def extract_match_scores(query, match):
    q = strip_non_alphanumeric(query).lower()
    m = strip_non_alphanumeric(match).lower()
    return {
        'token_set_ratio': fuzz.token_set_ratio(q, m),
        # 'ratio': fuzz.ratio(q, m),
        'token_sort_ratio': fuzz.token_sort_ratio(q, m),
    }

## Parse Reddit .tsv and query YTMusic for Match

* Some reddit entries are albums, most are tracks
* lots of title 'scrubbing' before query to clean
* saves match and unmatched seperatly, caches query responses 
    * cache in above ytmusic api cell
* scores match using fuzz metrics
    * tries to automate passing macthes with score threshold


#### Last run 10/17/2021


In [12]:
log_every_n_matches = 1000
tail_n_entries = 10
fname_splitter = '_'
expected_fname_toks = 4
expected_cols = ['reddit', 'youtube_id', 'title', 'url', 'author', 'timestamp',
                 'description', 'likes', 'dislikes', 'num_comments', 'num_plays',
                 'is_media', 'thumb_url', 'num_views']

matched_entries = []
unmatched_entries = []
log_tsvs = sorted(glob.glob(os.path.join(reddit_log_path, '*.tsv')))
print(f'Found {len(log_tsvs)} reddit tsvs')
for i, tsv_file in enumerate(log_tsvs):
    name = os.path.splitext(os.path.basename(tsv_file))[0]
    toks = name.split(fname_splitter)
    if len(toks) != expected_fname_toks:
        print(
            f'skipping tsv without {expected_fname_toks} "{fname_splitter}" split toks: {tsv_file}')
        continue
    sub, agg, count, timestamp = toks
    df = pd.read_csv(tsv_file, sep='\t', index_col=0)
    print(f'\n\n({i}/{len(log_tsvs)})  Loaded {len(df)} entries with {len(df.columns)} columns from {name}')
    assert set(df.columns) == set(expected_cols)

    # Loop thru entries
    for entry in df.itertuples():
        if 'youtube' not in entry.url and 'soundcloud' not in entry.url:
            print(f' skipping, Unknown url source {entry.url}')
        title_key, is_album = scrub_title(entry.title)

        # Check cache for saved YTMusic query response or previous match failures
        if entry.url in yt_res_cache:
            match = yt_res_cache[entry.url]
        elif title_key in yt_unmatched_cache:
            print(f'Skipping: previously unmatched entry: {title_key}')
            unmatched_entries.append(yt_unmatched_cache[title_key])
            continue
        # Query YTMusic
        else:
            match = {}
            try:
                if is_album:
                    res = ytm.search(query=title_key, filter='albums', limit=1)
                    if len(res):
                        res = res[0]
                        match['is_album'] = True
                        match['ytmusic_title'] = ''
                        match['ytmusic_album'] = res.get('title', '')
                        match['ytmusic_albumId'] = res.get('browseId', '')
                        if 'artist' in res:
                            match['ytmusic_artist'] = res['artists'][0].get('name', '')
                        match['ytmusic_key'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_album', '')}".lower()
                        match['ytmusic_playlist_id'] = f"sub-{match.get('ytmusic_albumId','')}"

                else:
                    res = ytm.search(query=title_key, filter='songs', limit=1)
                    if len(res):
                        res = res[0]
                        match['is_album'] = False
                        match['ytmusic_album'] = ''
                        if 'album' in res:
                            match['ytmusic_album'] = res['album'].get('name', '')
                            match['ytmusic_albumId'] = res['album'].get('id', '')
                        if 'artist' in res:
                            match['ytmusic_artist'] = res['artists'][0].get('name', '')
                        match['ytmusic_title'] = res.get('title', '')
                        match['ytmusic_videoId'] = res.get('videoId', '')
                        match['ytmusic_key'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_title', '')}".lower()
                        match['ytmusic_playlist_id'] = f"sub-{match.get('ytmusic_videoId','')}"

            except Exception as e:
                print(f'Error with {entry.title}: {e}')
                pass

            # Store unmatched ytmusic query
            if len(res) == 0:
                unmatch = {}
                unmatch['reddit_title'] = entry.title
                unmatch['reddit_source_url'] = entry.url
                unmatch['reddit_key'] = title_key
                unmatch['reddit_sub'] = sub
                unmatch['reddit_sub_id'] = f'{sub}-{title_key}'
                unmatch['reddit_aggregator'] = agg
                unmatched_entries.append(unmatch)
                yt_unmatched_cache[title_key] = unmatch
                print('Skipping : {entry.title}')


            # Other ytmusic fields
            match['ytmusic_artistId'] = res['artists'][0]['id']
            match['ytmusic_entry'] = f"{match.get('ytmusic_artist', '')} - {match.get('ytmusic_title', '')} - {match.get('ytmusic_album', '')}"
            match['ytmusic_duration'] = res['duration']
            match['ytmusic_year'] = res['year']
            match['ytmusic_resultType'] = res['resultType']
            yt_res_cache[entry.url] = match

        # Reddit fields
        match['reddit_title'] = entry.title
        match['reddit_key'] = title_key
        match['reddit_sub'] = sub
        match['reddit_aggregator'] = agg
        match['reddit_source_url'] = entry.url
        match['youtube_videoId'] = entry.youtube_id

        # Match Score
        scores = extract_match_scores(
            match['reddit_key'], match['ytmusic_key'])
        for sk, sv in scores.items():
            match[f'match_score_{sk}'] = sv
        # Thresholds based on first round of manual quality check
        # see: https://docs.google.com/spreadsheets/d/1CVAlR9pJ5wgu2eM8Aml2YVG573pnRdPKM69PsMNIMKk/edit#gid=854683398
        # see: reddit_scraper\logs\ytmusic\*.png
        if scores['token_sort_ratio'] < 40 or scores['token_set_ratio'] < 70:
            match['match_quality'] = 0.
        elif scores['token_sort_ratio'] > 75 or scores['token_set_ratio'] > 90:
            match['match_quality'] = 1.
        else:
            match['match_quality'] = 0.5

        # Update match results
        matched_entries.append(match)

        # if len(matched_entries) % log_every_n_matches == 0:
        #     cols = ['reddit_sub','reddit_aggregator','reddit_key', 'ytmusic_key', 'ytmusic_resultType']
        #     display(pd.DataFrame(matched_entries)[cols].tail(tail_n_entries))

# Process Matched entries
match_df = pd.DataFrame(matched_entries)
orig_len = len(match_df)
match_df = match_df.drop_duplicates(subset='ytmusic_playlist_id', keep='first')
print(f'Dropped {orig_len - len(match_df)} subreddit match duplicates')
print(f'\n\nSaving output tsvs to {reddit_log_path}')
match_file = os.path.join(reddit_log_path, 'ytmusic',
                          f'reddit_ytmusic_scored_matches_{date.today()}.tsv')
match_df.to_csv(match_file, sep='\t', header=True)

# Process Unmatched entries
unmatch_df = pd.DataFrame(unmatched_entries)
orig_len = len(unmatch_df)
unmatch_df = unmatch_df.drop_duplicates(subset='reddit_sub_id', keep='first')
print(f'Dropped {orig_len - len(unmatch_df)} subreddit unmatch duplicates')
unmatch_file = os.path.join(
    reddit_log_path, 'ytmusic', f'reddit_ytmusic_failed_matches_{date.today()}.tsv')
unmatch_df.to_csv(unmatch_file, sep='\t', header=True)


Found 39 reddit tsvs


Loaded 503 entries with 14 columns from 90shiphop_top-all_1000_1564979404
Skipping track: Mos Def -1998 -Mos Def & Talib Kweli - Respiration
Skipping track: The Notorious B.I.G. - "20 Years: Excess & Success" (A Tribute by Honeygold)
Skipping track: Straight Outta Compton Official Red Band Trailer #1 (2015) - Paul Giamatti Movie HD
Skipping track: wu tang clan demo tape 1992 ( complete ) ripped by stellar


Loaded 386 entries with 14 columns from blues_top-all_1000_1564979329
Skipping track: Harmonicas, Serendipity, and Satan: Oxford, MS by Revivalism
Skipping track: You See Me Laughin': the last of the hill country bluesmen  (Mississippi Blues documentary - 2002)
Skipping track: Laszlo Buring - Leavin' blues by LaszloRB
Skipping track: Hush Hush (Guitar Lightnin' Lee) by Wade Hilts
Skipping track: Crossroad blues on dobro -Oliver Jüchems
Skipping track: Gary Clark Jr. - "Don't Owe You a Thang" @ Sweetlife Festival, Columbia Md. Live HQ


Loaded 644 entries with 

## Create YTMusic Playlists from matches

* manually marked each match as 'ok' (pass) or 'x' (fail)
    * see: https://docs.google.com/spreadsheets/d/1CVAlR9pJ5wgu2eM8Aml2YVG573pnRdPKM69PsMNIMKk/edit#gid=854683398
* computed match score using a few fuzz algos
    * set basic threshold to try to automate grading

### Next Steps
* handle passing albums (similar cell as to tracks but for albums playlists)
* handle failed matches, att them to the unmaatched tsv with their scores and extra cols intact
    * maybe retry search? look and see if search query can be done better
    * if source is youtube add youtube link to correct ytmusic sub playlist (last resort)
        * add to albums vs track playist based on duration? 



### Done
* use fuzzy id scoring to determine if match is passing
    * if failing add to unmatched entries
    * if passing add to ytmusic {r.subreddit} playlist
        * have one playlist for albums and another for tracks
* split out 'likes' for each {r.subreddit} ytmusic playlist

#### Last run 10/17/2021

In [4]:
graded_tsv = os.path.join(reddit_log_path, 'ytmusic',
                    'reddit_ytmusic_scored_matches_and_graded_2021-10-16.tsv')
graded_matches = pd.read_csv(graded_tsv, sep='\t', index_col=0)

# Drop duplicate entries
# graded_matches.loc[graded_matches.is_album==True, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_videoId
# graded_matches.loc[graded_matches.is_album==False, 'ytmusic_playlist_id'] = graded_matches.reddit_sub + '-' + graded_matches.ytmusic_albumId

# Split pass / fail
passing = graded_matches.loc[graded_matches.manual_check == 'ok']
failing = graded_matches.loc[graded_matches.manual_check == 'x']


In [6]:
N=5
LIMIT = 900
passing_tracks = passing.loc[passing.is_album == False]
completed = []
completed = ['90shiphop', 'blues', 'chillmusic', 'chillwave', 'futurebass', 'futurebeats', 'futurefunkairlines', 'futuregarage', 'hiphop', 'hiphop101', 'hiphopheads', 'indie', 'indieheads', 'indierock', 'jazz', 'jazzyhiphop', 'lofihiphop', 'nudisco', 'psychedelicrock', 'rap', 'realdubstep', 'reggae', 'shoegaze', 'treemusic']
# Note: r/song doesnt work probably too big? > 1000 tracks? maybe not
for sub, df in passing_tracks.groupby('reddit_sub'):
    if sub in completed:
        continue

    # Create YTMusic Playlist from subreddit entries
    print(f'\nGenerating r/{sub} ytmusic playlist for {len(df)} passing tracks from ')
    vids = df.ytmusic_videoId.unique().tolist()
    title=f'x_r/{sub}_tracks'
    desc = f'Matched {len(vids)} tracks from r/{sub} using filters: {df.reddit_aggregator.unique()}'
    pl_id = ytm.create_playlist(title=title,  description=desc, privacy_status='PRIVATE', video_ids=vids)
    print(f'Saved {len(vids)} r/{sub} tracks playlist with id: {pl_id}, waiting {N} seconds...')
    time.sleep(N)

    # Fetch Just Saved Playlist
    metadata = ytm.get_playlist(pl_id, limit=LIMIT)
    tracks, metadata = parse_ytmusic_playlist(ytm, metadata)

    # Create Like subset
    liked_tracks = tracks.loc[tracks['likeStatus'] == 'LIKE']
    vids = liked_tracks.videoId.unique().tolist()
    desc = f'Liked subset of {len(vids)} entries from: {desc}'
    liked_pl_id = ytm.create_playlist(title=f'{title}_like', description=desc, video_ids=vids, privacy_status='PRIVATE')
    print(f'Filtered {len(vids)} LIKE r/{sub} tracks playlist with id: {liked_pl_id}, waiting {N} seconds...')
    time.sleep(N)
    
    # Create Unrated Radio subset
    unrated_tracks = tracks.loc[tracks['likeStatus'] != 'LIKE']
    vids = unrated_tracks.videoId.unique().tolist()
    desc = f'Unrated radio subset of {len(vids)} entries from: {desc}'
    radio_pl_id = ytm.create_playlist(title=f'{title}_radio', description=desc, video_ids=vids, privacy_status='PRIVATE')
    print(f'Filtered {len(vids)} INDIFFERENT r/{sub} radio tracks playlist with id: {liked_pl_id}, waiting {N} seconds...')
    time.sleep(N)

    # Delete original playlist now thet like/unrated is split
    ytm.delete_playlist(pl_id)
    print(f'Deleted r/{sub} tracks playlist with id: {pl_id}')
    completed.append(sub)
    



Generating r/90shiphop ytmusic playlist for 312 passing tracks from 

Generating r/blues ytmusic playlist for 208 passing tracks from 

Generating r/chillmusic ytmusic playlist for 463 passing tracks from 

Generating r/chillwave ytmusic playlist for 472 passing tracks from 

Generating r/futurebass ytmusic playlist for 438 passing tracks from 

Generating r/futurebeats ytmusic playlist for 653 passing tracks from 

Generating r/futurefunkairlines ytmusic playlist for 463 passing tracks from 

Generating r/futuregarage ytmusic playlist for 278 passing tracks from 

Generating r/hiphop ytmusic playlist for 764 passing tracks from 

Generating r/hiphop101 ytmusic playlist for 4 passing tracks from 

Generating r/hiphopheads ytmusic playlist for 165 passing tracks from 

Generating r/indie ytmusic playlist for 457 passing tracks from 

Generating r/indieheads ytmusic playlist for 75 passing tracks from 

Generating r/indierock ytmusic playlist for 349 passing tracks from 

Generating r/j